# 01 - Data Exploration

This notebook explores the COMPWALK-ACL and UCI Multivariate Gait datasets
used for ACL injury risk prediction.

## Datasets
1. **COMPWALK-ACL**: 92 participants (52 healthy, 40 ACL-injured) with IMU gait kinematics
2. **UCI Gait**: 10 healthy subjects, 3 conditions (unbraced, knee brace, ankle brace)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.compwalk_loader import build_dataset, load_metadata
from src.data.uci_loader import load_uci_data

sns.set_theme(style='whitegrid')
%matplotlib inline

## Load COMPWALK-ACL Dataset

In [ ]:
# Load the dataset
dataset = build_dataset()
print(f"Total records: {len(dataset)}")
print(f"Participants: {dataset['participant_id'].nunique()}")
print(f"\nCohort distribution:")
print(dataset.groupby('cohort')['participant_id'].nunique())
print(f"\nLabel distribution:")
print(dataset.groupby('label')['participant_id'].nunique())

In [ ]:
# Metadata statistics
metadata = load_metadata()
if not metadata.empty:
    print("Participant metadata:")
    display(metadata.describe())
else:
    print("No metadata file found.")

## Visualize Joint Angle Time Series

In [ ]:
# Compare knee flexion between healthy and ACL-injured
gait_pct = np.linspace(0, 100, 101)

for joint in ['knee_flexion', 'hip_flexion', 'ankle_dorsiflexion']:
    fig, ax = plt.subplots(figsize=(10, 5))
    
    for label, color, name in [(0, '#4CAF50', 'Healthy'), (1, '#F44336', 'ACL Injured')]:
        joint_data = dataset[(dataset['joint'] == joint) & (dataset['label'] == label)]
        if joint_data.empty:
            continue
        
        angles = np.vstack(joint_data['angle_timeseries'].values)
        mean = np.mean(angles, axis=0)
        std = np.std(angles, axis=0)
        
        ax.plot(gait_pct, mean, color=color, lw=2, label=f'{name} (n={len(angles)})')
        ax.fill_between(gait_pct, mean - std, mean + std, color=color, alpha=0.15)
    
    ax.set_xlabel('Gait Cycle (%)', fontsize=12)
    ax.set_ylabel('Angle (degrees)', fontsize=12)
    ax.set_title(f'{joint.replace("_", " ").title()}: Healthy vs ACL Injured', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## UCI Gait Dataset

In [ ]:
uci_data = load_uci_data()
if not uci_data.empty:
    print(f"UCI records: {len(uci_data)}")
    print(f"Subjects: {uci_data['participant_id'].nunique()}")
    print(f"Conditions: {uci_data['speed'].unique()}")
    print(f"Joints: {uci_data['joint'].unique()}")
else:
    print("UCI data not loaded. Run download first.")

## Summary Statistics

In [ ]:
# Summary table
summary = []
for joint in dataset['joint'].unique():
    for label in [0, 1]:
        subset = dataset[(dataset['joint'] == joint) & (dataset['label'] == label)]
        if subset.empty:
            continue
        angles = np.vstack(subset['angle_timeseries'].values)
        summary.append({
            'Joint': joint,
            'Group': 'Healthy' if label == 0 else 'ACL Injured',
            'N': len(angles),
            'Mean': np.mean(angles),
            'Std': np.std(angles),
            'Range Mean': np.mean(np.ptp(angles, axis=1)),
        })

summary_df = pd.DataFrame(summary)
display(summary_df)